# Feature Engineering: ECFP4 + Clean Descriptors

**Purpose:** Generate binding-relevant features for pChEMBL prediction

**Problem with previous features:**
- Dominated by ADMET/toxicity predictions (downstream, not causal)
- Missing structural fingerprints (substructure encoding)
- Result: R² 0.20 (barely better than mean)

**New feature set:**
- **Morgan Fingerprints (ECFP4):** 2048 bits encoding molecular substructure
- **Clean RDKit Descriptors:** ~25 physchem properties (MW, LogP, HBD, etc.)
- **Total:** ~2073 features (vs previous 78)

**Expected improvement:** R² 0.20 → 0.55-0.70

---

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# RDKit imports
import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Lipinski, Crippen, rdMolDescriptors
from rdkit.Chem.Scaffolds import MurckoScaffold

# Scipy for sparse matrices
from scipy.sparse import csr_matrix, save_npz
import matplotlib.pyplot as plt
import seaborn as sns

print("✅ Libraries imported successfully")
print(f"   RDKit version: {rdkit.__version__}")

## 2. Load Data

In [ ]:
# Load the 42k training dataset
data_path = Path("../chembl_curated_out/molecules_42k_with_bioactivity_training.csv")
df = pd.read_csv(data_path)

print(f"✅ Dataset loaded: {len(df):,} molecules")
print(f"   Columns: {len(df.columns)}")
print(f"\n📊 Target variable (pChEMBL):")
print(f"   Range: {df['pchembl_mean'].min():.2f} - {df['pchembl_mean'].max():.2f}")
print(f"   Mean: {df['pchembl_mean'].mean():.2f} ± {df['pchembl_mean'].std():.2f}")

# Check for SMILES
if 'canonical_smiles' not in df.columns:
    raise ValueError("❌ 'canonical_smiles' column not found!")

print(f"\n✅ SMILES column found")
print(f"   Sample: {df['canonical_smiles'].iloc[0][:50]}...")

## 3. Define Clean Descriptor Set

**Selected descriptors:** Binding-relevant physchem properties only

In [ ]:
def calculate_clean_descriptors(mol):
    """
    Calculate clean RDKit descriptors relevant to binding affinity.
    
    Categories:
    - Size/Shape: MW, Heavy atoms, Csp3
    - Lipophilicity: LogP, MR
    - H-bonding: HBD, HBA
    - Polarity: TPSA
    - Complexity: Rotatable bonds, Rings, Aromatic rings
    - Charge: Formal charge
    """
    if mol is None:
        return {}
    
    descriptors = {
        # Size and shape
        'MW': Descriptors.MolWt(mol),
        'HeavyAtomCount': Lipinski.HeavyAtomCount(mol),
        'FractionCsp3': Lipinski.FractionCSP3(mol),
        
        # Lipophilicity
        'LogP': Crippen.MolLogP(mol),
        'MolMR': Crippen.MolMR(mol),  # Molar refractivity
        
        # H-bonding
        'HBD': Lipinski.NumHDonors(mol),
        'HBA': Lipinski.NumHAcceptors(mol),
        
        # Polarity
        'TPSA': Descriptors.TPSA(mol),
        
        # Complexity
        'NumRotatableBonds': Lipinski.NumRotatableBonds(mol),
        'NumRings': Lipinski.RingCount(mol),
        'NumAromaticRings': Lipinski.NumAromaticRings(mol),
        'NumSaturatedRings': Lipinski.NumSaturatedRings(mol),
        'NumAliphaticRings': Lipinski.NumAliphaticRings(mol),
        
        # Heteroatoms
        'NumHeteroatoms': Lipinski.NumHeteroatoms(mol),
        
        # Charge
        'FormalCharge': Chem.GetFormalCharge(mol),
        
        # Aromatic proportion
        'AromaticProportion': len([atom for atom in mol.GetAtoms() if atom.GetIsAromatic()]) / mol.GetNumAtoms(),
        
        # SP3 carbons
        'NumSP3Carbons': sum([1 for atom in mol.GetAtoms() if atom.GetAtomicNum() == 6 and atom.GetHybridization() == Chem.HybridizationType.SP3]),
        
        # Bridgehead atoms
        'NumBridgeheadAtoms': rdMolDescriptors.CalcNumBridgeheadAtoms(mol),
        'NumSpiroAtoms': rdMolDescriptors.CalcNumSpiroAtoms(mol),
        
        # Additional complexity
        'BertzCT': Descriptors.BertzCT(mol),  # Complexity
        'Chi0v': Descriptors.Chi0v(mol),  # Connectivity index
        'Chi1v': Descriptors.Chi1v(mol),
        'Chi2v': Descriptors.Chi2v(mol),
        'Kappa1': Descriptors.Kappa1(mol),  # Shape index
        'Kappa2': Descriptors.Kappa2(mol),
    }
    
    return descriptors

print("✅ Descriptor function defined")
print(f"   Total descriptors: 25")

## 4. Generate Morgan Fingerprints (ECFP4) + Descriptors

**ECFP4 Parameters:**
- Radius: 2 (equivalent to ECFP4)
- nBits: 2048 (standard for ML)
- useFeatures: False (atom-type based, not pharmacophore)

In [ ]:
def generate_features(smiles):
    """
    Generate ECFP4 (2048 bits) + clean descriptors from SMILES.
    
    Returns:
        tuple: (fp_array, descriptors_dict, scaffold) or (None, None, None) if invalid
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, None, None
    
    # Generate Morgan fingerprint (ECFP4) as uint8 array
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)
    fp_array = np.array(fp, dtype=np.uint8)
    
    # Generate descriptors
    descriptors = calculate_clean_descriptors(mol)
    
    # Get Bemis-Murcko scaffold for scaffold split
    try:
        scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
    except:
        scaffold = smiles  # Fallback to original SMILES if scaffold fails
    
    return fp_array, descriptors, scaffold

# Test on first molecule
test_smiles = df['canonical_smiles'].iloc[0]
test_fp, test_desc, test_scaffold = generate_features(test_smiles)

if test_fp is not None:
    print(f"✅ Feature generation working")
    print(f"   ECFP4 shape: {test_fp.shape} (stored as uint8 array)")
    print(f"   Descriptors: {len(test_desc)}")
    print(f"   Scaffold: {test_scaffold[:50]}...")
    print(f"\n   Sample descriptors:")
    for key in list(test_desc.keys())[:5]:
        print(f"      {key}: {test_desc[key]:.3f}")
else:
    print("❌ Feature generation failed on test molecule")

## 5. Process All Molecules

**This will take ~2-5 minutes for 42k molecules**

In [ ]:
%%time

print("⏳ Processing 42,113 molecules...")
print("   This takes ~2-5 minutes\n")

# Generate features for all molecules
fingerprints_list = []
descriptors_list = []
scaffolds_list = []
failed_indices = []

for idx, smiles in tqdm(enumerate(df['canonical_smiles']), total=len(df), desc="Generating features"):
    fp, desc, scaffold = generate_features(smiles)
    
    if fp is None:
        failed_indices.append(idx)
        # Store NaN placeholders for failed molecules
        fingerprints_list.append(np.zeros(2048, dtype=np.uint8))
        descriptors_list.append({key: np.nan for key in ['MW', 'LogP', 'HBD', 'HBA', 'TPSA']})
        scaffolds_list.append(None)
    else:
        fingerprints_list.append(fp)
        descriptors_list.append(desc)
        scaffolds_list.append(scaffold)

# Convert to arrays
fingerprints_array = np.vstack(fingerprints_list)  # Shape: (42113, 2048)
descriptors_df = pd.DataFrame(descriptors_list)

# Store scaffolds
scaffolds_series = pd.Series(scaffolds_list, name='scaffold')

print(f"\n✅ Feature generation complete!")
print(f"   Success: {len(df) - len(failed_indices):,} molecules")
print(f"   Failed: {len(failed_indices)} molecules")
print(f"   Fingerprints shape: {fingerprints_array.shape}")
print(f"   Descriptors shape: {descriptors_df.shape}")


## 6. Combine with Metadata and Target

**Keep only:**
- Identifiers: molecule_chembl_id, canonical_smiles
- Target: pchembl_mean
- New features: ECFP4 + descriptors

In [ ]:
# Combine metadata + target + descriptors + scaffolds
metadata_cols = ['molecule_chembl_id', 'canonical_smiles', 'pchembl_mean']
metadata_df = df[metadata_cols].copy()

# Concatenate metadata + descriptors + scaffolds
final_df = pd.concat([
    metadata_df.reset_index(drop=True), 
    descriptors_df.reset_index(drop=True),
    scaffolds_series.reset_index(drop=True)
], axis=1)

print(f"✅ Final dataset created:")
print(f"   DataFrame shape: {final_df.shape} (metadata + descriptors + scaffold)")
print(f"   Fingerprints array: {fingerprints_array.shape} (stored separately)")
print(f"\n   Column breakdown:")
print(f"      Metadata: 3 (ID, SMILES, target)")
print(f"      Descriptors: 26")
print(f"      Scaffold: 1")
print(f"      DataFrame total: {len(final_df.columns)}")
print(f"      Fingerprints: 2048 (stored as numpy array)")

# Check for NaNs in descriptors
nan_count = final_df.isnull().sum().sum()
if nan_count > 0:
    print(f"\n⚠️  Warning: {nan_count} NaN values found in descriptors")
    # Don't drop - keep indices aligned, will handle in training
else:
    print(f"\n✅ No NaN values found")

## 7. Verify Feature Quality

In [ ]:
# Get descriptor columns
descriptor_cols = [col for col in final_df.columns 
                   if col not in ['molecule_chembl_id', 'canonical_smiles', 'pchembl_mean', 'scaffold']]

print("🔍 Feature Quality Check:")

# 1. ECFP4 sparsity
ecfp_sparsity = (fingerprints_array == 0).sum() / fingerprints_array.size
print(f"\n1. ECFP4 sparsity: {ecfp_sparsity:.2%}")
print(f"   (Expected: 85-95% zeros for biological molecules)")
print(f"   Average bits set per molecule: {fingerprints_array.sum(axis=1).mean():.1f}")

# 2. Check for zero-variance descriptors
zero_var_descriptors = [col for col in descriptor_cols if final_df[col].nunique() == 1]
print(f"\n2. Zero-variance descriptors: {len(zero_var_descriptors)}")
if len(zero_var_descriptors) > 0:
    print(f"   {zero_var_descriptors}")

# 3. Descriptor statistics
print(f"\n3. Descriptor summary (top 10):")
print(final_df[descriptor_cols].describe().T[['mean', 'std', 'min', 'max']].head(10))

# 4. Scaffold diversity
scaffold_counts = final_df['scaffold'].value_counts()
print(f"\n4. Scaffold diversity:")
print(f"   Unique scaffolds: {final_df['scaffold'].nunique():,}")
print(f"   Molecules per scaffold (mean): {len(final_df) / final_df['scaffold'].nunique():.1f}")
print(f"   Largest scaffold cluster: {scaffold_counts.iloc[0]} molecules")
print(f"   Scaffolds with 1 molecule: {(scaffold_counts == 1).sum():,}")

## 8. Sanity Checks: Target Distribution and Null Baseline

In [ ]:
# 1. Target distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(final_df['pchembl_mean'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].axvline(final_df['pchembl_mean'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {final_df["pchembl_mean"].mean():.2f}')
axes[0].set_xlabel('pChEMBL Value')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Target Distribution (pChEMBL)')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Box plot
axes[1].boxplot(final_df['pchembl_mean'], vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue', color='steelblue'),
                medianprops=dict(color='red', linewidth=2))
axes[1].set_ylabel('pChEMBL Value')
axes[1].set_title('Target Distribution (Box Plot)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("📊 Target Statistics:")
print(f"   Mean: {final_df['pchembl_mean'].mean():.3f}")
print(f"   Std: {final_df['pchembl_mean'].std():.3f}")
print(f"   Median: {final_df['pchembl_mean'].median():.3f}")
print(f"   Range: [{final_df['pchembl_mean'].min():.2f}, {final_df['pchembl_mean'].max():.2f}]")

# 2. Null baseline (always predict mean)
null_baseline_rmse = final_df['pchembl_mean'].std()
print(f"\n🎯 Null Baseline RMSE (predict mean): {null_baseline_rmse:.3f}")
print(f"   This is what R² = 0 looks like")
print(f"   Your model MUST beat this!")

# 3. Data leakage check
print(f"\n🔍 Data Leakage Check:")
leakage_keywords = ['pchembl', 'activity', 'assay', 'target', 'bioactivity']
leakage_cols = [col for col in final_df.columns if any(kw in col.lower() for kw in leakage_keywords)]
leakage_cols = [col for col in leakage_cols if col != 'pchembl_mean']  # Exclude target
if leakage_cols:
    print(f"   ⚠️  Warning: Potential leakage columns detected:")
    for col in leakage_cols:
        print(f"      - {col}")
else:
    print(f"   ✅ No obvious data leakage detected")

# 4. ECFP4 bit usage distribution
bit_usage = fingerprints_array.sum(axis=0)  # How many molecules have each bit set
print(f"\n🔬 ECFP4 Bit Usage:")
print(f"   Bits never used: {(bit_usage == 0).sum()} / 2048")
print(f"   Bits used in >50% molecules: {(bit_usage > len(final_df)*0.5).sum()}")
print(f"   Most common bit count: {bit_usage.max()} molecules")

## 9. Save New Training Dataset

In [ ]:
# Save descriptors + metadata as CSV
csv_path = Path("../chembl_curated_out/molecules_42k_ecfp4_features.csv")
final_df.to_csv(csv_path, index=False)
csv_size_mb = csv_path.stat().st_size / (1024 ** 2)

# Save fingerprints as compressed sparse matrix (MUCH more efficient)
fp_sparse = csr_matrix(fingerprints_array)
fp_path = Path("../chembl_curated_out/fingerprints_ecfp4_sparse.npz")
save_npz(fp_path, fp_sparse)
fp_size_mb = fp_path.stat().st_size / (1024 ** 2)

# Also save dense fingerprints as numpy array (for compatibility)
fp_dense_path = Path("../chembl_curated_out/fingerprints_ecfp4_dense.npy")
np.save(fp_dense_path, fingerprints_array)
fp_dense_size_mb = fp_dense_path.stat().st_size / (1024 ** 2)

print(f"✅ Datasets saved:")
print(f"\n   📄 CSV (descriptors + metadata):")
print(f"      Path: {csv_path}")
print(f"      Size: {csv_size_mb:.1f} MB")
print(f"      Shape: {final_df.shape}")
print(f"\n   🗜️  Fingerprints (sparse CSR):")
print(f"      Path: {fp_path}")
print(f"      Size: {fp_size_mb:.1f} MB ({fp_size_mb/fp_dense_size_mb*100:.1f}% of dense)")
print(f"      Shape: {fp_sparse.shape}")
print(f"      Sparsity: {ecfp_sparsity:.2%}")
print(f"\n   📦 Fingerprints (dense numpy):")
print(f"      Path: {fp_dense_path}")
print(f"      Size: {fp_dense_size_mb:.1f} MB")
print(f"      Shape: {fingerprints_array.shape}")

print(f"\n🎉 Feature engineering complete!")
print(f"\n📊 What changed:")
print(f"   OLD: 78 features (ADMET/toxicity noise dominated)")
print(f"   NEW: 2074 features (2048 ECFP4 + 26 clean descriptors)")
print(f"   Storage: Fingerprints stored as sparse matrix (efficient!)")
print(f"\n📈 Expected performance:")
print(f"   OLD: R² ≈ 0.20, RMSE ≈ 1.09 (broken features)")
print(f"   NEW: R² ≈ 0.55-0.70, RMSE ≈ 0.60-0.75 (proper binding features)")
print(f"\n⚡ Recommended models:")
print(f"   • HistGradientBoostingRegressor (fast, no scaling needed)")
print(f"   • XGBoost/LightGBM (best for sparse features)")
print(f"   • Random Forest (baseline)")
print(f"\n🧪 Scaffold split ready: Use 'scaffold' column for realistic generalization")
print(f"\n➡️  Next: Update training notebook with scaffold split + HistGradientBoosting")

## 10. Compare Old vs New Feature Sets

In [ ]:
# Load old dataset for comparison
old_df = pd.read_csv("../chembl_curated_out/molecules_42k_with_bioactivity_training.csv")

# Define what we removed
old_features = [col for col in old_df.columns if col not in ['molecule_chembl_id', 'canonical_smiles', 
                                                                'pchembl_mean', 'pchembl_median', 'pchembl_std',
                                                                'pchembl_min', 'pchembl_max', 'has_bioactivity',
                                                                'is_active', 'is_potent', 'is_highly_potent',
                                                                'activity_count', 'target_count', 'assay_count',
                                                                'target_names', 'first_approval', 'max_phase',
                                                                'pref_name', 'bemis_murcko_scaffold']]

# Identify ADMET features (likely those we removed)
admet_keywords = ['tox', 'DILI', 'AMES', 'hERG', 'CYP', 'BBB', 'Caco', 'Pgp', 'HIA', 
                  'FDA', 'Clearance', 'T12', 'VD', 'Nephro', 'Hepato', 'Cardio', 'Oto']
admet_features = [col for col in old_features if any(keyword.lower() in col.lower() for keyword in admet_keywords)]

print("📊 Feature Set Comparison:")
print(f"\n🔴 OLD (REMOVED):")
print(f"   Total old features: {len(old_features)}")
print(f"   ADMET/toxicity features: {len(admet_features)}")
print(f"   Example removed features:")
for feat in admet_features[:10]:
    print(f"      ❌ {feat}")

print(f"\n🟢 NEW (ADDED):")
print(f"   ECFP4 fingerprint bits: 2048")
print(f"   Clean RDKit descriptors: 26")
print(f"   Example new descriptors:")
for feat in descriptor_cols[:10]:
    print(f"      ✅ {feat}")

print(f"\n💡 Why this matters:")
print(f"   • ECFP4 encodes substructure → directly relevant to binding")
print(f"   • ADMET/toxicity → downstream effects, not causal to affinity")
print(f"   • Clean descriptors → interpretable, physically meaningful")